# Star / galaxy separation in ground-based surveys using *Euclid* as a reference

This notebook reproduces the method described in Thomas, Simon, Gwyn et al. 2026,
*"UNIONS and Euclid: Calibrating point source -- galaxy separation for
ground-based surveys"*. The main difference is that this code use the public DR1 MER Final Catalogue, whereas the original paper measures Euclid photometry directly with SExtractor on the VIS-pipeline stacked tiles on the Euclid Data Release 1 (2026). 

A neural network (ANN) is trained to separate point sources (stars) from
extended sources (galaxies) in a ground-based, multi-band imaging survey
(e.g. UNIONS/CFIS), using labels derived from the higher-resolution
*Euclid* VIS imaging.

**Public release notes**

- The ground-based photometric catalogue used to train the network is
  **proprietary** and is *not* included in this repository. You must supply
  your own catalogue with an equivalent set of columns (see the mapping
  table right after the data is read in).
- Column names from both the private survey and the public *Euclid* catalogue
  are renamed to a common, self-explanatory naming scheme immediately after
  the data is loaded, so that the rest of the notebook never depends on the
  original (survey-specific) column names.


In [1]:
from astropy.table import Table, vstack, hstack
from astropy.coordinates import SkyCoord
from astropy import units as u

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import MultipleLocator, FormatStrFormatter
from matplotlib.colors import BoundaryNorm

import pickle
import random

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras import backend as K
from keras.callbacks import EarlyStopping

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_curve, roc_auc_score

import shap

plt.rc('font', size=14)
plt.rc('axes', titlesize=14)
plt.rc('axes', labelsize=14)
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)
plt.rc('legend', fontsize=14)
plt.rc('figure', titlesize=16)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.fontset"] = "dejavuserif"
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'


/home/gthomas/.local/lib/python3.8/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (3.0.4)/charset_normalizer (3.0.1) doesn't match a supported version!
  warnings.warn(


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 1. Column-name configuration

All column renaming is centralised here so that:

- the **private** ground-survey column names never leak into the rest of
  the notebook (only the generic names on the right-hand side are used
  below), and
- the *Euclid* part can be swapped for any MER-catalogue-like table (DR1
  public release, later data releases, etc.) by editing this cell only.

>  **TODO for the user**: edit `GROUND_COLUMN_MAP` so that the keys match the
>  exact column names of *your own* (private) ground-based catalogue. The
>  values (right-hand side) are the generic names used throughout the rest
>  of the notebook and do not need to change.


In [ ]:
# --------------------------------------------------------------------
# Ground-based (PRIVATE) survey -> generic column names
# --------------------------------------------------------------------
# Adjust the keys (left) to match your own reduction pipeline's column
# names. Only the values (right) are used from here on.
GROUND_COLUMN_MAP = {
    "ALPHA_J2000":    "RA",              # source right ascension  [deg]
    "DELTA_J2000":    "DEC",             # source declination      [deg]
    "field":          "TILE_ID",         # tile / pointing identifier
    "MAG_COG":        "MAG_TOTAL",       # curve-of-growth (total) magnitude
    "MAGERR_COG":     "MAGERR_TOTAL",    # error on the total magnitude
    "PREDIQ":         "SEEING",          # predicted image quality (FWHM) [arcsec]
    "A_WORLD":        "SEMI_MAJOR_AXIS", # source semi-major axis [deg]
    "B_WORLD":        "SEMI_MINOR_AXIS", # source semi-minor axis [deg]
}
# Fixed-aperture magnitudes: MAG_APER_1..20 / MAGERRR_APER_1..20 (note the
# typo "MAGERRR" present in some pipeline outputs) are mapped automatically.
N_GROUND_APERTURES = 20
for i in range(1, N_GROUND_APERTURES + 1):
    GROUND_COLUMN_MAP[f"MAG_APER_{i}"] = f"MAG_APER_{i}"
    GROUND_COLUMN_MAP[f"MAGERRR_APER_{i}"] = f"MAGERR_APER_{i}"

# --------------------------------------------------------------------
# Euclid Q1 MER Final Catalogue (PUBLIC) -> generic column names
# --------------------------------------------------------------------
# Reference: Euclid Q1 MER Photometry Cookbook
# https://euclid.esac.esa.int/dr/q1/dpdd/merdpd/merphotometrycookbook.html
# Fluxes are given in micro-Jansky; the AB zero point below converts them
# to magnitudes: m_AB = -2.5*log10(flux_uJy) + ZP_AB
ZP_AB = 23.9

EUCLID_ID_COLUMN_MAP = {
    "OBJECT_ID":        "EUC_OBJECT_ID",
    "RIGHT_ASCENSION":  "EUC_RA",
    "DECLINATION":      "EUC_DEC",
    "VIS_DET":          "EUC_VIS_DET",
    "DET_QUALITY_FLAG": "EUC_QUALITY_FLAG",
    "POINT_LIKE_PROB":  "EUC_POINT_LIKE_PROB",   # Euclid's own star/galaxy probability (not used as a feature here)
}

# Q1 only provides 4 fixed PSF-matched apertures (n x FWHM of the worst-seeing band)
EUCLID_APERTURES = [1, 2, 3, 4]


## 2. Load the ground-based catalogue and the Euclid DR1 photometry

Rename the ground-survey columns immediately after reading, so that everything downstream uses only the generic names defined above. The Euclid DR1 fluxes (already cross-matched into this table -- see note in the code cell below) are converted from micro-Jansky to AB magnitudes right away as well.


In [ ]:
# TODO: point this to your own (private) reduction catalogue.
data = Table.read("/path/to/private/ground_survey_catalog.fits")

# Rename columns right after reading -- from here on we never use the
# original, pipeline-specific column names again.
for old_name, new_name in GROUND_COLUMN_MAP.items():
    if old_name in data.colnames and old_name != new_name:
        data.rename_column(old_name, new_name)

print("Ground survey columns (after renaming):")
print(data.colnames)



# --------------------------------------------------------------------
# Convert Euclid Q1 VIS fluxes (micro-Jansky) to AB magnitudes
# --------------------------------------------------------------------
# NOTE: this cell assumes `data` already contains the Euclid DR1 MER columns,
# i.e. the private catalogue and the public Euclid DR1 catalogue have already
# been cross-matched upstream (see the notebook history / previous version
# for a worked example of the cross-match with astropy SkyCoord if you still
# need to do this step).
ZP_AB = 23.9  # AB zero point for MER fluxes given in micro-Jansky

# Total magnitude: FLUX_DETECTION_TOTAL is the Kron total flux (the "good
# old FLUX_AUTO" of SExtractor) -- this is what the I_E,Kron label in the
# diagram below refers to. Using FLUX_VIS_PSF here instead would bias the
# total magnitude of extended (non point-like) sources, since a PSF fit
# systematically underestimates the flux of resolved galaxies.
flux_total = np.asarray(data["FLUX_DETECTION_TOTAL"])
with np.errstate(divide="ignore", invalid="ignore"):
    data["EUC_MAG"] = np.where(
        flux_total > 0, -2.5 * np.log10(flux_total) + ZP_AB, np.nan
    )

# Fixed n x FWHM aperture magnitudes and their errors
for n in EUCLID_APERTURES:
    flux = np.asarray(data[f"FLUX_VIS_{n}FWHM_APER"])
    fluxerr = np.asarray(data[f"FLUXERR_VIS_{n}FWHM_APER"])
    with np.errstate(divide="ignore", invalid="ignore"):
        data[f"EUC_MAG_APER_{n}"] = np.where(
            flux > 0, -2.5 * np.log10(flux) + ZP_AB, np.nan
        )
        data[f"EUC_MAGERR_APER_{n}"] = np.where(
            flux > 0, (2.5 / np.log(10)) * (fluxerr / flux), np.nan
        )

## 3. Quality cuts

Remove sources with unreliable photometry or non-finite shape parameters.


In [ ]:
data = data[(data["EUC_MAG"] < 30) & (data["EUC_MAG"] > 0)]
data = data[(data["MAG_TOTAL"] < 28) & (data["MAG_TOTAL"] > 0)
            & (data["MAGERR_TOTAL"] > 0) & (data["MAGERR_TOTAL"] < 1.0)]

for n in EUCLID_APERTURES:
    data = data[data[f"EUC_MAGERR_APER_{n}"] < 1.0]

for i in range(1, N_GROUND_APERTURES + 1):
    data = data[data[f"MAGERR_APER_{i}"] < 1.0]

data = data[np.isfinite(data["SEMI_MAJOR_AXIS"])
            & np.isfinite(data["SEMI_MINOR_AXIS"])
            & np.isfinite(data["SEEING"])]

print("Number of sources after quality cuts:", len(data))
data.write("ground_euclid_matched_clean.fits", overwrite=True)


## 4. Define star / galaxy labels from the *Euclid* size-magnitude diagram

Point sources form a tight, well-defined locus in the *Euclid*
`EUC_MAG_APER_2 - EUC_MAG_APER_4` vs `EUC_MAG_AUTO` plane. Sources inside
this locus (and brighter than `EUC_MAG_AUTO = 26`) are labelled as stars
(`label = 1`); everything else is labelled as a galaxy (`label = 0`).


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 3.5))

xbins = np.arange(15.0, 28.001, 0.05)
ybins = np.arange(0.0, 1.5, 0.01)

_, _, _, sc = ax.hist2d(
    data["EUC_MAG"],
    data["EUC_MAG_APER_2"] - data["EUC_MAG_APER_4"],
    bins=(xbins, ybins), cmap="Greys", norm="log", rasterized=True,
)

# Empirical star locus boundary
ax.plot(xbins[xbins <= 26.001],
        -0.0001 * (xbins[xbins <= 26.001] - 20.5) ** 4 + 0.35, "r-", lw=1)
ax.axvline(x=26.0, ymin=0.12 / 1.5,
           ymax=(-0.0001 * (26.0 - 20.5) ** 4 + 0.35) / 1.5, color="red", lw=1)
ax.axhline(y=0.12, xmax=(26.0 - 15) / (28.0 - 15), color="red", lw=1)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label(r"N$_\mathrm{obj}\,/\,$bin")
cbar.solids.set_alpha(1)

ax.set_xlim(15, 28)
ax.set_ylim(0.0, 1.5)
ax.set_xlabel(r"$I_{\mathrm{E}, \mathrm{Kron}}$")
ax.set_ylabel(r"$I_\mathrm{E,2}-I_\mathrm{E,4}$")

fig.tight_layout()
plt.savefig("StarGal_euclid.pdf", format="pdf")
plt.show()


In [ ]:
# label = 1 -> point source (star), label = 0 -> extended source (galaxy)
data["label"] = np.where(
    (data["EUC_MAG_APER_2"] - data["EUC_MAG_APER_4"]
     < -0.0001 * (data["EUC_MAG"] - 20.5) ** 4 + 0.35)
    & (data["EUC_MAG_APER_2"] - data["EUC_MAG_APER_4"] > 0.12)
    & (data["EUC_MAG"] < 26.0),
    1, 0,
)

print("Stars:", (data["label"] == 1).sum(), "  Galaxies:", (data["label"] == 0).sum())


## 5. Train / test split

An independent 20% hold-out test set is set aside before any scaling or
model fitting.


In [ ]:
n = len(data)
test_indices = np.random.choice(n, size=int(0.2 * n), replace=False)
mask = np.zeros(n, dtype=bool)
mask[test_indices] = True

test = data[mask].copy()
train = data[~mask].copy()

print(f"Train: {len(train)}   Test: {len(test)}")


## 6. Build the feature matrix and scale the inputs

Features are the ground-based fixed-aperture magnitudes, the total
magnitude, the seeing, and the source shape (semi-major/minor axes).


In [ ]:
feature_names = ([f"MAG_APER_{i}" for i in range(1, N_GROUND_APERTURES + 1)]
                 + ["MAG_TOTAL", "SEEING", "SEMI_MAJOR_AXIS", "SEMI_MINOR_AXIS"])

def build_feature_matrix(table):
    cols = [table[name] for name in feature_names]
    return np.vstack(cols).T

X = build_feature_matrix(train)
X_test = build_feature_matrix(test)

y = np.array(train["label"])
y_test = np.array(test["label"])

print("Feature matrix shape:", X.shape)

# Encode labels as one-hot vectors
le = LabelEncoder()
y_enc = le.fit_transform(y)
num_classes = len(np.unique(y_enc))
y_cat = keras.utils.to_categorical(y_enc, num_classes=num_classes)

y_test_enc = le.transform(y_test)  # transform only, fit on train
y_test_cat = keras.utils.to_categorical(y_test_enc, num_classes=num_classes)

# Standardize features (fit on train only)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)


## 7. Define and train the ANN

A simple fully-connected network with a 3-stage decaying learning-rate
schedule and early stopping on the validation loss.


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.1),
    layers.Dense(num_classes, activation="softmax"),
])
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

# Training parameters
epochs = 500
batch_size = 2000
learning_rates = [1e-3, 1e-4, 1e-5]

early_stop = EarlyStopping(monitor="val_loss", mode="auto", verbose=1,
                            patience=20, restore_best_weights=True)
callbacks = [early_stop]

epoch_change_lr = np.zeros(len(learning_rates) - 1)
loss_ANN, valloss_ANN = None, None

for idx, lr in enumerate(learning_rates):
    print("Learning rate: %.g" % lr)
    K.set_value(model.optimizer.learning_rate, lr)

    if idx > 0:
        history = model.fit(
            X_scaled, y_cat, validation_split=0.2, callbacks=callbacks,
            epochs=epochs, batch_size=batch_size, verbose=1,
            initial_epoch=int(epoch_change_lr[idx - 1]),
        )
    else:
        history = model.fit(
            X_scaled, y_cat, validation_split=0.2, callbacks=callbacks,
            epochs=epochs, batch_size=batch_size, verbose=1,
        )

    if idx < len(learning_rates) - 1:
        hist_loss = history.history["loss"][:-early_stop.patience - 1]
        hist_valloss = history.history["val_loss"][:-early_stop.patience - 1]
        epoch_change_lr[idx] = history.epoch[-early_stop.patience]
    else:
        hist_loss = history.history["loss"]
        hist_valloss = history.history["val_loss"]

    if idx > 0:
        loss_ANN = np.concatenate([loss_ANN, hist_loss])
        valloss_ANN = np.concatenate([valloss_ANN, hist_valloss])
    else:
        loss_ANN = np.array(hist_loss)
        valloss_ANN = np.array(hist_valloss)

# Save the training history to disk
history_table = np.column_stack([
    np.arange(1, len(loss_ANN) + 1), loss_ANN, valloss_ANN
])
np.savetxt("training_loss_history.txt", history_table,
           header="epoch train_loss val_loss", delimiter=" ")

plt.plot(loss_ANN, "-", label="Train")
plt.plot(valloss_ANN, "-", label="Validation")
for epoch in epoch_change_lr:
    plt.axvline(x=epoch, ls="--", color="k")
legend = plt.legend(scatterpoints=1, loc="upper right", fontsize=12)
legend.get_frame().set_edgecolor("none")
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.yscale("log")
plt.show()
plt.close()

# Predictions on the training and test sets
y_pred_probs = model.predict(X_scaled)
y_pred_labels = le.inverse_transform(np.argmax(y_pred_probs, axis=1))

y_test_pred_probs = model.predict(X_test_scaled)
y_test_pred_labels = le.inverse_transform(np.argmax(y_test_pred_probs, axis=1))


## 8. Save the trained ANN, the scaler, and the labelled datasets


In [ ]:
# --- Save the trained model and the feature scaler ---
model.save("star_galaxy_ann.keras")
with open("scaler_star_galaxy_ann.pkl", "wb") as f:
    pickle.dump(scaler, f)
print("ANN and scaler saved.")

# --- Save the training and test sets, including model predictions ---
train["pred_label"] = y_pred_labels
train["proba_galaxy"] = y_pred_probs[:, 0]
train["proba_star"] = y_pred_probs[:, 1]
train.write("train_set_with_predictions.fits", overwrite=True)

test["pred_label"] = y_test_pred_labels
test["proba_galaxy"] = y_test_pred_probs[:, 0]
test["proba_star"] = y_test_pred_probs[:, 1]
test.write("test_set_with_predictions.fits", overwrite=True)
print("Train/test sets with predictions saved.")


## 9. Quick sanity check: ROC curve on the test set


In [ ]:
y_true_binary = (y_test_enc == 1).astype(int)
y_score = y_test_pred_probs[:, 1]

fpr, tpr, _ = roc_curve(y_true_binary, y_score)
auc = roc_auc_score(y_true_binary, y_score)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random classifier")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve on the independent test set (class 1 = star)")
plt.legend()
plt.grid()
plt.show()


## 10. Explainability: SHAP feature importance

SHAP (SHapley Additive exPlanations) values quantify how much each input
feature contributes to the network's prediction, for the "star" class.


In [ ]:
# Background reference sample used by the SHAP explainer
background = X_scaled[np.random.choice(X_scaled.shape[0], 200, replace=False)]
explainer = shap.GradientExplainer(model, background)

# Compute SHAP values on a subsample of the test set (full test set can be slow)
X_sample = X_test_scaled[:500]
shap_values = explainer.shap_values(X_sample)

STAR_CLASS_INDEX = 1  # index of the "star" class in the softmax output
shap.summary_plot(shap_values[STAR_CLASS_INDEX], X_sample, feature_names=feature_names)


In [ ]:
# Global feature importance: mean(|SHAP value|), averaged over classes and samples
shap_stack = np.stack(shap_values, axis=0)  # (num_classes, n_samples, n_features)
mean_abs_shap_global = np.mean(np.abs(shap_stack), axis=(0, 1))
importance_pct_global = 100 * mean_abs_shap_global / mean_abs_shap_global.sum()

df_importance = pd.DataFrame({
    "feature": feature_names,
    "importance_%": importance_pct_global,
}).sort_values("importance_%", ascending=False).reset_index(drop=True)

print(df_importance)

fig, ax = plt.subplots(figsize=(12, 5))
x_pos = np.arange(len(feature_names))
bars = ax.bar(x_pos, importance_pct_global, color="steelblue", edgecolor="black", linewidth=0.5)

ax.set_xticks(x_pos)
ax.set_xticklabels(feature_names, rotation=90, fontsize=10)
ax.set_ylabel("Relative information content (%)", fontsize=12)
ax.set_xlabel("Feature", fontsize=12)
ax.set_title("Relative importance of each input feature (SHAP)", fontsize=13)

for i, v in enumerate(importance_pct_global):
    ax.text(i, v + 0.3, f"{v:.1f}%", ha="center", va="bottom", fontsize=8)

ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("feature_importance.pdf", bbox_inches="tight", dpi=300)
plt.show()
